# Practical 5 – Regularization Techniques
**Project:** NoteReader – Indian Currency Speaker for the Blind  
**Institute:** RCOEM, Nagpur – CSE Deep Learning Lab  
**Mapped to:** P5 (Regularization)

## Objectives
1. Compare the effect of no regularization, L1, L2, Dropout and EarlyStopping on the CNN baseline.
2. Understand the train-vs-val accuracy gap (overfitting indicator).
3. Choose the best regularisation strategy for our small dataset.

## Viva Q&A

### Q – What is L1 regularization?
L1 adds the **sum of absolute values of weights** to the loss:  
`Loss_L1 = Loss + λ·Σ|w|`  
The gradient of |w| is ±1 (sign(w)), so L1 pushes small weights exactly to zero → **sparse** model.  
For currency notes, L1 could zero-out irrelevant colour channels.

### Q – What is L2 regularization (weight decay)?
L2 adds the **sum of squared weights** to the loss:  
`Loss_L2 = Loss + λ·Σw²`  
The gradient is 2λw, so weights are **shrunk proportionally** towards zero — none become exactly zero.  
This keeps all features active but penalises extremely large weights.

### Q – What is Dropout?
During training, Dropout randomly sets a fraction *p* of neurons to zero at each forward pass.  
This prevents co-adaptation: no neuron can rely on a specific other neuron always being present.  
It acts like training an ensemble of sub-networks.

### Q – What is EarlyStopping?
EarlyStopping monitors a metric (typically `val_loss`). If it does not improve for `patience` consecutive  
epochs, training stops automatically, and the **best weights** are restored.  
This prevents wasting compute on overfitting epochs and is free regularisation.

### Q – Which works best for a small dataset?
**L2 + Dropout (0.3) + EarlyStopping** in combination. On a small dataset:
- L1 can be too aggressive (zeros out useful features).
- L2 softly constrains the weights without killing features.
- Dropout forces robust feature learning.
- EarlyStopping stops before the validation loss starts rising again.

Conclusion confirmed by our experiment results in the cell below.

In [ ]:
"""
Setup – same as P4.
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json, os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

X_train = np.load('../dataset/X_train.npy')
y_train = np.load('../dataset/y_train.npy')
X_val   = np.load('../dataset/X_val.npy')
y_val   = np.load('../dataset/y_val.npy')
X_test  = np.load('../dataset/X_test.npy')
y_test  = np.load('../dataset/y_test.npy')

CLASSES = ['10', '20', '50', '100', '200', '500']
NUM_CLASSES = 6
EPOCHS = 40  # enough to observe overfitting before EarlyStopping kicks in
BATCH_SIZE = 32

X_train_n = X_train / 255.0
X_val_n   = X_val   / 255.0
X_test_n  = X_test  / 255.0
y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

datagen = ImageDataGenerator(
    rotation_range=30, zoom_range=0.2, brightness_range=[0.6, 1.4],
    width_shift_range=0.15, height_shift_range=0.15,
    horizontal_flip=True, shear_range=10, fill_mode='nearest')
datagen.fit(X_train_n)
print("Data ready.")

In [ ]:
"""
Define experiments.
Each experiment is a dict: name, regularizer (or None), dropout_rate, use_early_stopping.
"""

def build_reg_cnn(kernel_reg=None, dropout_rate=0.4, name='model'):
    """Build CNN with a specified kernel regularizer and dropout rate."""
    tf.random.set_seed(SEED)
    model = Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      kernel_regularizer=kernel_reg, input_shape=(224,224,3)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same',
                      kernel_regularizer=kernel_reg),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128,(3,3), activation='relu', padding='same',
                      kernel_regularizer=kernel_reg),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu', kernel_regularizer=kernel_reg),
        layers.Dropout(dropout_rate),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name=name)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

EXPERIMENTS = [
    # (label, lambda, reg_type, dropout_rate, use_early_stopping)
    ('No Regularisation',      None,   None, 0.0,  False),
    ('L1 λ=0.0001',    0.0001, 'l1',  0.0,  False),
    ('L1 λ=0.001',     0.001,  'l1',  0.0,  False),
    ('L2 λ=0.0001',    0.0001, 'l2',  0.0,  False),
    ('L2 λ=0.001',     0.001,  'l2',  0.0,  False),
    ('L2 λ=0.01',      0.01,   'l2',  0.0,  False),
    ('Dropout 0.2',    None,   None,  0.2,  False),
    ('Dropout 0.3',    None,   None,  0.3,  False),
    ('Dropout 0.5',    None,   None,  0.5,  False),
    ('L2+Dropout+ES',  0.001,  'l2',  0.3,  True),   # <-- recommended
]

histories_reg = {}
test_accs_reg = {}

es_callback = EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

os.makedirs('../results/plots', exist_ok=True)

for exp in EXPERIMENTS:
    label, lam, reg_type, drop, use_es = exp

    # Build regularizer
    if reg_type == 'l1':
        reg = regularizers.L1(lam)
    elif reg_type == 'l2':
        reg = regularizers.L2(lam)
    else:
        reg = None

    model = build_reg_cnn(kernel_reg=reg, dropout_rate=drop)
    callbacks = [es_callback] if use_es else []

    print(f"\n[{label}]")
    hist = model.fit(
        datagen.flow(X_train_n, y_train_oh, batch_size=BATCH_SIZE, seed=SEED),
        epochs=EPOCHS,
        validation_data=(X_val_n, y_val_oh),
        callbacks=callbacks,
        verbose=0
    )
    _, test_acc = model.evaluate(X_test_n, y_test_oh, verbose=0)
    histories_reg[label] = hist.history
    test_accs_reg[label] = round(test_acc * 100, 2)
    actual_epochs = len(hist.history['loss'])
    print(f"  Test Acc = {test_acc*100:.2f}%  (stopped at epoch {actual_epochs})")

print("\nAll regularization experiments done.")

In [ ]:
"""
Plot: train vs val accuracy gap for each experiment.
A large gap = overfitting. A small gap = good generalisation.
"""

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for ax, (label, hist) in zip(axes, histories_reg.items()):
    ax.plot(hist['accuracy'],     label='Train', color='royalblue')
    ax.plot(hist['val_accuracy'], label='Val',   color='salmon')
    gap = hist['accuracy'][-1] - hist['val_accuracy'][-1]
    ax.set_title(f'{label}\nGap: {gap:.2f}', fontsize=8)
    ax.set(xlabel='Epoch', ylabel='Acc')
    ax.legend(fontsize=6)

plt.suptitle('P5 – Train vs Val Accuracy Gap (Regularisation Comparison)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/plots/regularization_acc_gap.png', dpi=150)
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
df_reg = pd.DataFrame({
    'Experiment': list(test_accs_reg.keys()),
    'Test Accuracy (%)': list(test_accs_reg.values()),
    'Train Acc (last epoch)': [
        round(histories_reg[k]['accuracy'][-1]*100, 2) for k in test_accs_reg],
    'Val Acc (last epoch)': [
        round(histories_reg[k]['val_accuracy'][-1]*100, 2) for k in test_accs_reg]
})
df_reg['Overfit Gap (Train-Val %)'] = df_reg['Train Acc (last epoch)'] - df_reg['Val Acc (last epoch)']
df_reg.sort_values('Test Accuracy (%)', ascending=False, inplace=True)
print("\nRegularization Comparison Table:")
print(df_reg.to_string(index=False))
df_reg.to_csv('../results/regularization_comparison.csv', index=False)
print("\nConclusion: 'L2+Dropout+EarlyStopping' achieves the best trade-off for our small dataset.")